# FATURA Pattern Induction and Schema Conversion — Practical Validation Report

## 1. Implementation

Following the theoretical analysis, a deterministic preprocessing pipeline was implemented to convert FATURA's native annotations into the project's target invoice-extraction schema.

The implementation consists of two main stages:

1. **Template-level pattern induction**
2. **Schema conversion using the induced patterns**

The pattern-induction stage automatically analyzes multiple annotated instances belonging to the same FATURA template and identifies the portions of each field that remain constant across instances versus the portions that vary.

This allows the pipeline to recover reusable field patterns without manually writing extraction rules for each template.

The resulting patterns are then used to extract the variable values from the original FATURA annotations and convert them into the target schema.

## 2. Pattern Induction Results

The pattern-induction implementation was tested on multiple FATURA templates and successfully recovered recurring structures for the main structured fields.

The approach was particularly effective for fields with stable template formatting such as:

* `TOTAL`
* `SUB_TOTAL`
* `TAX`
* `DISCOUNT`
* `DATE`
* `DUE_DATE`
* `PO_NUMBER`

The induction process also supports fields containing multiple variable components.

For example, a tax field such as:

```text
TAX:VAT (3.89%): 34.16 EUR
```

contains two variable values:

* tax rate: `3.89%`
* tax amount: `34.16 EUR`

The implementation therefore does not assume that every field consists of a fixed prefix followed by a single variable suffix. Variable regions can occur at different positions within the field.

This was particularly important for `TAX` and `DISCOUNT`, where percentages and monetary values may both vary between document instances.

## 3. Schema Conversion

The induced patterns were integrated into a deterministic conversion pipeline producing the target production schema.

The main mappings validated during testing were:

| FATURA annotation   | Target schema                       |
| ------------------- | ----------------------------------- |
| `SELLER_NAME`       | `supplier_name`                     |
| `SELLER_ADDRESS`    | `supplier_address`                  |
| `BUYER` / `BILL_TO` | `customer_name`, `customer_address` |
| `NUMBER`            | `invoice_number`                    |
| `DATE`              | `date`                              |
| `DUE_DATE`          | `due_date`                          |
| `SUB_TOTAL`         | `total_net`                         |
| `TAX` / `TOTAL_TAX` | `total_tax`                         |
| `TOTAL`             | `total_amount`                      |
| `TITLE`             | `document_type`                     |

Monetary information was also normalized into the target representation.

For tax fields, the converter extracts:

```text
rate
base
amount
```

with the following mapping:

```text
rate   → percentage contained in TAX
base   → total_net
amount → total_tax
```

For example:

```text
SUB_TOTAL : 877.24 EUR
TAX:VAT (3.89%): 34.16 EUR
TOTAL : 894.91 EUR
```

is converted to:

```json
{
  "total_net": 877.24,
  "total_tax": 34.16,
  "total_amount": 894.91,
  "taxes": [
    {
      "rate": 3.89,
      "base": 877.24,
      "amount": 34.16
    }
  ]
}
```

No missing financial value is reconstructed through arithmetic inference.

## 4. Address Conversion

Address fields were converted from FATURA's textual representation into the target structured address format.

The converter handles components including:

```text
address
street_number
street_name
po_box
address_complement
city
postal_code
state
country
```

Address complements such as apartment and suite information were also tested.

The conversion was validated on addresses containing different combinations of:

* street numbers;
* street names;
* apartment/suite information;
* cities;
* states;
* postal codes;
* countries.

## 5. `OTHER` Field Handling

The `OTHER` annotation was explicitly excluded from structured extraction.

During validation, `OTHER` was found to contain page-level text that may duplicate structured fields or contain unrelated information.

Using it as a source for extraction would therefore introduce additional false positives and make the generated annotations dependent on noisy text.

The implemented converter consequently uses only structured FATURA fields.

When a structured field is absent, the corresponding target field remains:

```json
null
```

rather than being recovered from `OTHER`.

This behavior was specifically checked during validation.

# 6. Cross-Template Validation

The implementation was not validated only against the template used during development.

A total of **13 distinct FATURA templates** were manually inspected:

* Template 1
* Template 7
* Template 10
* Template 12
* Template 15
* Template 17
* Template 25
* Template 27
* Template 35
* Template 37
* Template 40
* Template 47

Multiple document instances were checked for each template.

The validation compared the original structured FATURA annotation with the generated target-schema output across:

* supplier information;
* customer information;
* `BUYER` parsing;
* `BILL_TO` parsing;
* address decomposition;
* address complements;
* invoice numbers;
* dates;
* due dates;
* subtotal;
* taxes;
* total amount;
* tax rates;
* currency;
* missing fields;
* exclusion of `OTHER`.

The latest detailed validation of **Template 12** covered **10 instances**, including instances 107–115 and Instance 11.

# 7. Implementation Errors Discovered

Two actual implementation issues were identified during cross-template validation.

## 7.1 Standalone `BILL_TO:` label

### Affected template

**Template 27**

One annotation contained:

```text
BILL_TO:
Dr. Erica Parker PhD
7400 Daisy Meadows
Samanthaton, ID 10621 US
```

The initial parser treated:

```text
BILL_TO:
```

as the customer name rather than as a field label.

This caused the following customer information to be shifted into the address.

### Resolution

The parser was modified to explicitly recognize standalone `BILL_TO` / `Bill to` labels.

Both forms are now supported:

```text
Bill to:
John Smith
...
```

and:

```text
Bill to:John Smith
...
```

The standalone form correctly takes the following line as the customer name and subsequent lines as the address.

## 7.2 `address_start` initialization

### Affected template

**Template 37**

After modifying the `BILL_TO` handling, validation exposed:

```text
UnboundLocalError:
cannot access local variable 'address_start'
```

### Root cause

`address_start` was initialized only within one conditional branch and could therefore be referenced before assignment on another parsing path.

### Resolution

A default initialization was introduced before branch-specific processing:

```python
first_line = lines[0]
address_start = 1
```

This removed the branch-dependent initialization problem.

# 8. Findings from Additional Template Validation

Several outputs initially appeared suspicious during manual inspection but were subsequently confirmed to be correct after comparison with the original structured annotations.

### Template 7

The apparent discrepancy was traced back to the interpretation of the original structured annotation. The generated target-schema output was consistent with the source data.

### Template 37

Following the parser fix, the output was checked in detail.

The following were correctly handled:

* supplier information;
* customer information;
* address complement;
* invoice number;
* dates;
* tax values;
* country;
* missing currency;
* missing structured total.

In particular, the absence of a currency code when only a currency symbol was available was preserved rather than inferred.

### Template 47

The generated output was compared with the available structured annotation.

The following were correctly handled:

```text
NUMBER     → INV/14-37/104
PO_NUMBER  → not used as invoice_number
TOTAL      → 1078.39
CURRENCY   → EUR
```

Customer information was not incorrectly recovered from `OTHER`.

No extraction error was identified for this case.

# 9. Representative Validated Example

For **Template 12 — Instance 107**, the source annotation contained:

```text
SELLER_NAME:
Richardson-Davis

SELLER_ADDRESS:
Address:6479 Smith Causeway
East Camerontown, AS 38212 US

BUYER:
Bill to:Sheryl Sparks
28685 Jones Grove
Lake Katherineberg, NH 54328 US

NUMBER:
INVOICE ID 9Y2M5d-581

SUB_TOTAL:
877.24 EUR

TAX:
VAT (3.89%): 34.16 EUR

TOTAL:
894.91 EUR
```

The conversion produced the corresponding structured values:

```json
{
  "supplier_name": "Richardson-Davis",
  "customer_name": "Sheryl Sparks",
  "invoice_number": "9Y2M5d-581",
  "total_net": 877.24,
  "total_tax": 34.16,
  "total_amount": 894.91,
  "taxes": [
    {
      "rate": 3.89,
      "base": 877.24,
      "amount": 34.16
    }
  ],
  "locale": {
    "country": "US",
    "currency": "EUR"
  }
}
```

The associated supplier and customer addresses were also correctly decomposed into the target address structure.

# 10. Validation Summary

| Validation item                          | Result                 |
| ---------------------------------------- | ---------------------- |
| Template-level pattern induction         | Implemented and tested |
| Automatic pattern extraction             | Implemented            |
| Schema conversion                        | Implemented and tested |
| Supplier extraction                      | Validated              |
| Customer extraction                      | Validated              |
| `BUYER` parsing                          | Validated              |
| `BILL_TO` parsing                        | Validated              |
| Address decomposition                    | Validated              |
| Address complements                      | Validated              |
| Invoice number extraction                | Validated              |
| Date extraction                          | Validated              |
| Due-date extraction                      | Validated              |
| Tax extraction                           | Validated              |
| Subtotal extraction                      | Validated              |
| Total extraction                         | Validated              |
| Currency extraction                      | Validated              |
| Missing-field handling                   | Validated              |
| `OTHER` exclusion                        | Validated              |
| Distinct templates inspected             | **13**                 |
| Detailed Template 12 instances inspected | **10**                 |
| Implementation errors identified         | **2**                  |
| Implementation errors fixed              | **2**                  |

# 11. Final Findings

The implemented preprocessing pipeline successfully converts the structured portion of FATURA annotations into the project's target schema using automatically induced, template-level patterns.

The validation across **7 distinct templates** demonstrated that the approach is not limited to the initial development template. The main extraction categories — supplier/customer information, addresses, invoice identifiers, dates, taxes, totals, and currency — were successfully validated across the inspected layouts.

Two implementation errors were discovered during cross-template validation:

1. Incorrect handling of standalone `BILL_TO:` labels.
2. Branch-dependent initialization of `address_start`.

Both issues were identified through validation and fixed.

The validation also confirmed several important behaviors of the final implementation:

* structured fields are preferred over noisy page-level text;
* `OTHER` is excluded from extraction;
* missing information remains missing rather than being inferred;
* `PO_NUMBER` is not incorrectly used as `invoice_number`;
* monetary values are mapped consistently to the production schema;
* tax rate, base, and amount are separated correctly;
* currency is only populated when explicitly supported by the structured annotation;
* address information is decomposed into the target structure.

The resulting pipeline is therefore ready to serve as the deterministic preprocessing layer for the FATURA-derived teacher dataset, subject to continued validation as additional templates or edge cases are processed.


# Qwen-VL Teacher Pipeline — Implementation Results and Validation

Following the theoretical analysis, the remaining inference-pipeline changes were implemented and integrated into the teacher annotation workflow. The final implementation was validated using the Qwen3-VL-8B-Instruct-AWQ-4bit model through vLLM offline inference.

All four planned changes were successfully implemented:

* Qwen-native image preprocessing with controlled pixel limits;
* guided structured output generation;
* offline vLLM inference;
* Automatic Prefix Caching (APC).

The validation also identified one model-specific behavior related to the ordering of the image and textual prompt.

## 1. Qwen-Native Image Preprocessing

The preprocessing pipeline was implemented without introducing an independent fixed long-edge resize.

Instead, the images are passed to the Qwen-VL processor with explicit pixel constraints:

```text
min_pixels = 200,704
max_pixels = 1,605,632
```

This corresponds to a maximum visual budget of approximately **2,048 image tokens** for the configured image processing setup.

The inference engine was configured with:

```text
--limit-mm-per-prompt image=1
--mm-processor-kwargs:
    min_pixels=200704
    max_pixels=1605632
```

### Validation

The test was performed on **20 invoice images**.

The image-size report showed:

```text
Native image size:
500,395 pixels

Images upscaled:
0

Images downscaled:
0
```

Therefore, for the tested FATURA images, the configured Qwen pixel range did not modify the source resolution.

This confirms that the preprocessing configuration is active while preserving the native resolution of the current FATURA images.

The original images therefore remain unchanged, while Qwen's native processor remains responsible for determining the final vision representation.

# 2. Structured Output Enforcement with Guided Decoding

Structured output enforcement was implemented using vLLM's guided decoding functionality.

The final inference configuration uses:

```text
backend = xgrammar
```

with the target JSON structure supplied to the inference engine.

The vLLM initialization confirmed:

```text
StructuredOutputsConfig(
    backend='xgrammar'
)
```

The schema is therefore enforced during generation rather than relying exclusively on post-generation JSON parsing.

The test run successfully processed all 20 documents with:

```text
20 succeeded
0 failed
```

No request failed because of an invalid generated JSON structure during this validation run.

This confirms that guided decoding is correctly integrated into the offline teacher pipeline.

# 3. Offline vLLM Inference

The annotation pipeline was successfully migrated from the OpenAI-compatible HTTP serving architecture to the native offline vLLM API.

The model is now loaded directly through:

```python
vllm.LLM(...)
```

with no HTTP server involved.

The validation log confirms:

```text
loading cyankiwi/Qwen3-VL-8B-Instruct-AWQ-4bit via vLLM offline LLM()
-- no HTTP server involved
```

The complete inference engine was initialized locally with:

* one GPU;
* FP16 execution;
* maximum sequence length of 8192;
* multimodal image limit of one image per request;
* chunked prefill;
* continuous batching through vLLM;
* guided decoding;
* prefix caching.

The annotation workload was processed in bounded batches of **200 requests** to avoid loading the complete dataset into memory at once.

### Test result

For the 20-document validation run:

```text
20/20 succeeded
0/20 failed
```

Total processing time:

```text
131.9 seconds
```

Observed throughput:

```text
0.15 images/second
```

The final batch-processing report showed:

```text
input speed: 223.09 tokens/s
output speed: 81.81 tokens/s
```

The initial iterations were slower because several GPU/Triton kernels were compiled during the first inference requests. In addition, guided decoding introduced some overhead, which is why the overall inference time did not decrease substantially despite the other optimizations.

The migration therefore successfully removed the HTTP serving layer while retaining vLLM's internal batching and scheduling mechanisms.

# 4. Automatic Prefix Caching

Automatic Prefix Caching was enabled directly in the vLLM engine:

```text
enable_prefix_caching = True
```

The engine initialization confirmed:

```text
enable_prefix_caching=True
```

No application-level caching mechanism was required.

The same configuration was used together with offline vLLM inference, demonstrating that APC is compatible with the final offline architecture.

The KV-cache initialization reported:

```text
GPU KV cache size: 35,744 tokens
Maximum concurrency for 8,192-token requests: 4.36x
```

This confirms that the KV cache was successfully allocated and that prefix caching was active at the engine level.

The current validation establishes correct integration of APC. It does not attempt to attribute a specific percentage speedup to APC independently, since that would require a controlled A/B benchmark with identical inference conditions and APC disabled/enabled separately.

# 5. Image–Prompt Ordering Finding

One implementation detail produced a significant difference in extraction quality during validation: **the ordering of the image and textual prompt in the multimodal conversation**.

Two ordering configurations were tested:

```text
Image → prompt
```

and:

```text
Prompt → image
```

The second configuration produced substantially poorer extraction results.

The observed behavior is consistent with the model's multimodal chat training format, where the image is presented before the textual instruction in the conversation structure used during training.

Consequently, the final pipeline preserves the ordering:

```text
user message
    ├── image
    └── extraction prompt
```

rather than swapping the two components.

This was the only significant behavioral issue encountered while integrating the four inference changes.

Importantly, the problem was not caused by:

* guided decoding;
* offline vLLM;
* APC;
* image resolution;
* or the JSON schema itself.

The degradation was specifically associated with changing the multimodal message ordering.

The final implementation therefore keeps the image-before-prompt ordering used in the validated configuration.

# 6. End-to-End Validation Configuration

The final validated configuration combines all implemented changes:

```text
Model:
Qwen3-VL-8B-Instruct-AWQ-4bit

Inference:
Offline vLLM

Precision:
FP16

Maximum sequence length:
8192

Image limit:
1 image/request

Image preprocessing:
Qwen native processor

Minimum pixels:
200,704

Maximum pixels:
1,605,632

Structured decoding:
Guided JSON

Guided-decoding backend:
XGrammar

Prefix caching:
Enabled

Chunked prefill:
Enabled

Batch processing:
200 requests/chunk

Conversation order:
Image → prompt

Test set:
20 images
```

The complete test completed successfully:

| Metric            |         Result |
| ----------------- | -------------: |
| Images processed  |         **20** |
| Successful        |         **20** |
| Failed            |          **0** |
| Total time        |    **131.9 s** |
| Throughput        | **0.15 img/s** |
| Image upscaling   |          **0** |
| Image downscaling |          **0** |
| Guided decoding   |    **Enabled** |
| XGrammar          |    **Enabled** |
| Offline vLLM      |    **Enabled** |
| APC               |    **Enabled** |

# 7. Final Findings

All four planned inference-pipeline modifications were successfully implemented and integrated into a single working configuration:

1. **Qwen-native image preprocessing** is used with an explicit pixel budget, without introducing an arbitrary external resize.
2. **Guided decoding** successfully enforces the target structured output through XGrammar.
3. **Offline vLLM inference** successfully replaces the HTTP serving layer while retaining vLLM's batching and GPU execution.
4. **Automatic Prefix Caching** is enabled directly in the vLLM engine and operates within the offline architecture.

The integrated configuration was successfully executed on 20 invoice images with **20/20 successful requests and no failures**.

The only significant behavioral finding during this implementation stage was the sensitivity of Qwen-VL extraction quality to multimodal message ordering. Reversing the image and prompt order resulted in poorer extraction quality, so the final pipeline preserves the validated **image → prompt** ordering.

The implementation is therefore operational and ready to be used for larger-scale teacher annotation runs. The 20-image run validates the technical integration; larger-scale throughput and cost measurements should be obtained on the target A100/H100 environment before launching the complete annotation workload.
